# Deep Past Challenge — Akkadian → English Translation
### Dual-Environment Pipeline: Train on Colab, Infer on Kaggle

**Model**: `google/flan-t5-xxl` (11B params) — instruction-tuned T5, strongest language understanding  
**Training**: QLoRA (4-bit NF4 + LoRA adapters) on **Google Colab L4** (24 GB VRAM)  
**Inference**: 8-bit quantization with `device_map="auto"` across **Kaggle 2× T4** (32 GB total)  
**Metric**: Geometric Mean(BLEU, chrF++) micro-averaged via SacreBLEU  

> **Workflow**: Train all 3 phases on Colab → merge LoRA → upload model to Kaggle as dataset → run inference-only on Kaggle.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============ Cell 1: Setup & Imports ============
import os, re, csv, math, random, warnings
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from typing import List, Dict, Tuple

warnings.filterwarnings("ignore")

# ---- Environment Detection ----
KAGGLE = False #os.path.exists('/kaggle/input')
COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')

if KAGGLE:
    DATA_DIR = Path('/kaggle/input/deep-past-initiative-machine-translation')
    OUTPUT_DIR = Path('/kaggle/working')
elif COLAB:
    DATA_DIR = Path('/content/drive/MyDrive/deep-past-initiative-machine-translation')   # Upload CSVs here or mount Drive
    OUTPUT_DIR = Path('/content/drive/MyDrive/deep-past-initiative-machine-translation')
    OUTPUT_DIR.mkdir(exist_ok=True)
else:
    DATA_DIR = Path('.')
    OUTPUT_DIR = Path('./output')
    OUTPUT_DIR.mkdir(exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ---- GPU Detection ----
N_GPUS = torch.cuda.device_count()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

env_name = 'Kaggle' if KAGGLE else ('Colab' if COLAB else 'Local')
print(f"Environment: {env_name}")
print(f"Data dir: {DATA_DIR}")
print(f"GPUs available: {N_GPUS}")
if N_GPUS > 0:
    for i in range(N_GPUS):
        name = torch.cuda.get_device_name(i)
        mem = torch.cuda.get_device_properties(i).total_mem / 1e9
        print(f"  GPU {i}: {name} ({mem:.1f} GB)")

if COLAB:
    print("Mode: TRAINING (QLoRA on Colab)")
elif KAGGLE:
    print("Mode: INFERENCE (8-bit on Kaggle 2×T4)")
else:
    print("Mode: LOCAL (dev/debug)")

## Preprocessing Functions
Clean Akkadian transliteration and English translations following competition guidelines.

In [ ]:
# ============ Cell 2: Preprocessing ============

def clean_transliteration(text: str) -> str:
    """Clean Akkadian transliteration following competition guidelines."""
    if not isinstance(text, str):
        return ""
    text = text.replace('Ḫ', 'H').replace('ḫ', 'h')
    for ch in ['!', '?', '/']:
        text = text.replace(ch, '')
    text = text.replace('˹', '').replace('˺', '')
    text = text.replace('[', '').replace(']', '')
    text = re.sub(r'<<(.+?)>>', r'\1', text)
    text = re.sub(r'<(?!gap|big_gap)(.+?)>', r'\1', text)
    text = re.sub(r'\{large break\}', '<big_gap>', text)
    text = re.sub(r'…+', '<big_gap>', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def clean_translation(text: str) -> str:
    """Clean English translation."""
    if not isinstance(text, str):
        return ""
    text = text.strip().strip('"').strip("'")
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def extract_syllables(text: str) -> list:
    """Extract syllables from transliterated Akkadian."""
    words = text.split()
    syllables = []
    for word in words:
        parts = word.split('-')
        syllables.extend(parts)
    return syllables

print("✅ Preprocessing functions defined")

## Load & Prepare Data
Load all 7 data sources and prepare training pairs.

In [ ]:
# ============ Cell 3: Load & Prepare Data ============

csv.field_size_limit(2**30)  # Handle large CSV fields (publications.csv)

# 1. Document-level training data
train_df = pd.read_csv(DATA_DIR / 'train.csv')
train_df['src'] = train_df['transliteration'].apply(clean_transliteration)
train_df['tgt'] = train_df['translation'].apply(clean_translation)
print(f"Document-level pairs: {len(train_df)}")

# 2. Sentence-level data from Sentences_Oare
sent_df = pd.read_csv(DATA_DIR / 'Sentences_Oare_FirstWord_LinNum.csv')
sent_pairs_raw = sent_df[sent_df['translation'].notna()].copy()
print(f"Sentence translations available: {len(sent_pairs_raw)}")

# 3. Lexicon lookup
lex_df = pd.read_csv(DATA_DIR / 'OA_Lexicon_eBL.csv')
form_to_lexeme = {}
for _, row in lex_df.iterrows():
    form = str(row.get('form', ''))
    lexeme = str(row.get('lexeme', ''))
    if form != 'nan' and lexeme != 'nan':
        form_to_lexeme[form] = lexeme
print(f"Lexicon entries: {len(form_to_lexeme)}")

# 4. Dictionary
dict_df = pd.read_csv(DATA_DIR / 'eBL_Dictionary.csv')
word_to_def = {}
for _, row in dict_df.iterrows():
    w = str(row.get('word', '')).strip()
    d = str(row.get('definition', '')).strip()
    if w != 'nan' and d != 'nan' and len(d) > 3:
        word_to_def[w.split(' ')[0].strip()] = d
print(f"Dictionary entries: {len(word_to_def)}")

# 5. Published texts (monolingual)
pub_df = pd.read_csv(DATA_DIR / 'published_texts.csv')
print(f"Published texts: {len(pub_df)}")

# 6. Test data
test_df = pd.read_csv(DATA_DIR / 'test.csv')
print(f"Test samples: {len(test_df)}")

# Sumerogram lookup
SUMEROGRAM_MAP = {
    'KÙ.BABBAR': 'silver', 'URUDU': 'copper', 'AN.NA': 'tin',
    'KÙ.GI': 'gold', 'TÚG': 'textile', 'ANŠE': 'donkey',
    'GÍN': 'shekel', 'GÚ': 'talent', 'ITU.KAM': 'month',
    'KIŠIB': 'seal', 'DUMU': 'son of', 'DINGIR': 'god',
    'É': 'house', 'SÍG': 'wool', 'ŠU.NÍGIN': 'total',
    'DAM.GÀR': 'merchant', 'DAM': 'wife', 'DUB': 'tablet',
    'TÚG.ḪI.A': 'textiles', 'ANŠE.ḪI.A': 'donkeys',
}

## Build Training Dataset with Sentence-Level Alignment
Critical: test is sentence-level, train is document-level. We align using Sentences_Oare overlap + document chunking.

In [ ]:
# ============ Cell 4: Build Training Pairs ============

all_pairs = []  # list of (source, target, task_type)

# ---- 1. Sentence Alignment from Sentences_Oare ↔ train.csv ----
# Find overlapping documents
train_ids = set(train_df['oare_id'].dropna().astype(str))
sent_ids = set(sent_pairs_raw['text_uuid'].dropna().astype(str))
overlap_ids = train_ids & sent_ids

print(f"Overlapping documents (train ↔ sentences): {len(overlap_ids)}")

def segment_transliteration_by_sentences(text_id, train_row, sentences_for_doc):
    """Use word positions from Sentences_Oare to segment transliteration."""
    pairs = []
    src_text = str(train_row['src'])
    src_words = src_text.split()
    
    sentences_sorted = sentences_for_doc.sort_values('first_word_number')
    
    for i, (_, sent_row) in enumerate(sentences_sorted.iterrows()):
        translation = str(sent_row.get('translation', ''))
        if translation == 'nan' or len(translation.strip()) < 5:
            continue
        
        first_word = int(sent_row.get('first_word_number', 0))
        # Estimate end: next sentence start or end of doc
        if i + 1 < len(sentences_sorted):
            next_start = int(sentences_sorted.iloc[i + 1].get('first_word_number', len(src_words)))
        else:
            next_start = len(src_words)
        
        if first_word < len(src_words):
            segment = ' '.join(src_words[first_word:min(next_start, len(src_words))])
            if len(segment.split()) >= 2:
                pairs.append((segment, clean_translation(translation), 'SENT'))
    
    return pairs

# Build sentence-aligned pairs
for text_id in overlap_ids:
    train_rows = train_df[train_df['oare_id'].astype(str) == text_id]
    sent_rows = sent_pairs_raw[sent_pairs_raw['text_uuid'].astype(str) == text_id]
    
    if len(train_rows) > 0 and len(sent_rows) > 0:
        pairs = segment_transliteration_by_sentences(
            text_id, train_rows.iloc[0], sent_rows
        )
        all_pairs.extend(pairs)

print(f"Sentence-aligned pairs: {len([p for p in all_pairs if p[2] == 'SENT'])}")

# ---- 2. Document-level pairs (chunked) ----
MAX_DOC_WORDS = 128

for _, row in train_df.iterrows():
    src, tgt = row['src'], row['tgt']
    src_words = src.split()
    tgt_words = tgt.split()
    
    if len(src_words) <= MAX_DOC_WORDS:
        all_pairs.append((src, tgt, 'DOC'))
    else:
        n_chunks = (len(src_words) + MAX_DOC_WORDS - 1) // MAX_DOC_WORDS
        s_chunk = len(src_words) // n_chunks
        t_chunk = len(tgt_words) // n_chunks if n_chunks > 0 else len(tgt_words)
        for i in range(n_chunks):
            cs = ' '.join(src_words[i*s_chunk : min((i+1)*s_chunk + 16, len(src_words))])
            ct = ' '.join(tgt_words[i*t_chunk : min((i+1)*t_chunk + 16, len(tgt_words))])
            if len(cs.split()) >= 5:
                all_pairs.append((cs, ct, 'DOC'))

# ---- 3. Word-level pairs from lexicon + dictionary ----
for form, lexeme in form_to_lexeme.items():
    base = lexeme.split(' ')[0].strip()
    defn = word_to_def.get(base, word_to_def.get(lexeme, ''))
    if defn and len(defn) > 3:
        clean_def = defn.split(';')[0].strip().strip('"').strip()
        if 3 < len(clean_def) < 100:
            all_pairs.append((form, clean_def, 'WORD'))

# Add Sumerogram pairs
for form, meaning in SUMEROGRAM_MAP.items():
    all_pairs.append((form, meaning, 'WORD'))

# ---- Summary ----
task_counts = {}
for _, _, t in all_pairs:
    task_counts[t] = task_counts.get(t, 0) + 1

print(f"\nTotal training pairs: {len(all_pairs)}")
for task, count in sorted(task_counts.items()):
    print(f"  {task}: {count}")

## Model: flan-t5-xxl (11B) + QLoRA

**Training (Colab L4)**: Load in 4-bit NF4 quantization (~5.5 GB) + LoRA adapters (~20 MB trainable). Gradient checkpointing keeps VRAM under 20 GB.  
**Inference (Kaggle 2×T4)**: Merge LoRA → load full model in 8-bit with `device_map="auto"` splitting across both GPUs (~5.5 GB/GPU).

On Kaggle (inference-only), we skip QLoRA setup and load the pre-merged model directly.

In [ ]:
# ============ Cell 5: Model Loading — flan-t5-xxl + QLoRA ============

from transformers import (
    AutoModelForSeq2SeqLM, AutoTokenizer, T5ForConditionalGeneration,
    Seq2SeqTrainingArguments, Seq2SeqTrainer,
    DataCollatorForSeq2Seq, EarlyStoppingCallback,
    BitsAndBytesConfig,
)
from datasets import Dataset
from sklearn.model_selection import train_test_split

# ---- Paths ----
MODEL_HF_NAME = "google/flan-t5-xxl"          # HuggingFace identifier (fallback)
# On Kaggle inference, load the pre-merged model uploaded as a dataset:
KAGGLE_MODEL_PATH = "/kaggle/input/flan-t5-xxl-akkadian/final_merged"

# ============================================================
# TRAINING PATH (Colab L4): QLoRA 4-bit + LoRA adapters
# ============================================================
if not KAGGLE:
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

    # --- Download model via kagglehub (faster than HF Hub on Colab) ---
    import kagglehub
    print("Downloading flan-t5-xxl from Kaggle Models via kagglehub...")
    local_model_path = kagglehub.model_download("google/flan-t5/pyTorch/xxl/1")
    print(f"Model downloaded to: {local_model_path}")

    # 4-bit NF4 quantization config — model fits in ~5.5 GB VRAM
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,   # nested quantization saves ~0.4 GB
    )

    print(f"Loading model in 4-bit NF4 from {local_model_path}...")
    tokenizer = AutoTokenizer.from_pretrained(local_model_path)
    model = T5ForConditionalGeneration.from_pretrained(
        local_model_path,
        quantization_config=bnb_config,
        device_map="auto",                # auto-place layers on available GPU(s)
        torch_dtype=torch.float16,
    )

    # Prepare for k-bit training (freeze base, cast layernorm to fp32)
    model = prepare_model_for_kbit_training(model)

    # Enable gradient checkpointing to save VRAM (trades compute for memory)
    model.gradient_checkpointing_enable()

    # LoRA configuration — target attention projections
    lora_config = LoraConfig(
        r=16,                              # rank
        lora_alpha=32,                     # scaling factor
        target_modules=["q", "v"],         # T5 attention projections
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.SEQ_2_SEQ_LM,
    )

    model = get_peft_model(model, lora_config)

    # Stats
    trainable, total = model.get_nb_trainable_parameters()
    print(f"Total parameters:     {total / 1e9:.2f}B")
    print(f"Trainable (LoRA):     {trainable / 1e6:.1f}M ({100 * trainable / total:.2f}%)")
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / 1e9
        print(f"GPU memory allocated: {alloc:.1f} GB")

# ============================================================
# INFERENCE PATH (Kaggle 2×T4): 8-bit with device_map split
# ============================================================
else:
    print(f"Loading pre-merged model from {KAGGLE_MODEL_PATH} in 8-bit...")
    tokenizer = AutoTokenizer.from_pretrained(KAGGLE_MODEL_PATH)
    model = T5ForConditionalGeneration.from_pretrained(
        KAGGLE_MODEL_PATH,
        load_in_8bit=True,
        device_map="auto",       # splits across 2×T4 automatically
        torch_dtype=torch.float16,
    )
    print(f"Model loaded across {N_GPUS} GPU(s)")
    if hasattr(model, 'hf_device_map'):
        devices_used = set(str(v) for v in model.hf_device_map.values())
        print(f"Devices used: {devices_used}")

## Phase 1: Vocabulary Pre-Training (Syllable → Meaning)
Train on WORD pairs first so the model learns Akkadian syllable patterns and Sumerogram meanings before tackling full sentences.

In [ ]:
# ============ Cell 6: Phase 1 — Vocabulary Pre-Training ============
# (Training only — skipped on Kaggle inference)

if not KAGGLE:
    # Separate WORD pairs from sentence/doc pairs
    word_pairs = [(s, t) for s, t, task in all_pairs if task == 'WORD']
    sent_doc_pairs = [(s, t, task) for s, t, task in all_pairs if task in ('SENT', 'DOC', 'MINE')]

    # Add task tags
    word_sources = [f"[WORD] {s}" for s, t in word_pairs]
    word_targets = [t for s, t in word_pairs]

    # Split
    w_train_src, w_val_src, w_train_tgt, w_val_tgt = train_test_split(
        word_sources, word_targets, test_size=0.05, random_state=SEED
    )
    print(f"Phase 1 — WORD pairs: {len(w_train_src)} train, {len(w_val_src)} val")

    # Tokenize
    MAX_SRC_P1, MAX_TGT_P1 = 128, 64

    def tokenize_p1(examples):
        inputs = tokenizer(examples['source'], max_length=MAX_SRC_P1, padding='max_length', truncation=True)
        with tokenizer.as_target_tokenizer():
            labels = tokenizer(examples['target'], max_length=MAX_TGT_P1, padding='max_length', truncation=True)
        inputs['labels'] = labels['input_ids']
        return inputs

    ds_p1_train = Dataset.from_dict({'source': w_train_src, 'target': w_train_tgt})
    ds_p1_val = Dataset.from_dict({'source': w_val_src, 'target': w_val_tgt})
    ds_p1_train = ds_p1_train.map(tokenize_p1, batched=True, remove_columns=['source', 'target'])
    ds_p1_val = ds_p1_val.map(tokenize_p1, batched=True, remove_columns=['source', 'target'])

    # Training
    collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

    # QLoRA on L4: model ~6GB in 4-bit, LoRA adapters ~20MB
    # WORD pairs are short (128 tok) → can use moderate batch
    P1_BATCH = 8
    P1_EVAL_BATCH = 16
    P1_GRAD_ACCUM = 4   # effective batch = 8 × 4 = 32

    p1_output_dir = str(OUTPUT_DIR / "ckpt_phase1")

    args_p1 = Seq2SeqTrainingArguments(
        output_dir=p1_output_dir,
        num_train_epochs=5,
        per_device_train_batch_size=P1_BATCH,
        per_device_eval_batch_size=P1_EVAL_BATCH,
        gradient_accumulation_steps=P1_GRAD_ACCUM,
        learning_rate=2e-4,            # higher LR for LoRA
        weight_decay=0.01,
        warmup_steps=200,
        lr_scheduler_type="cosine",
        eval_strategy="steps",
        eval_steps=500,
        save_strategy="steps",
        save_steps=500,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        predict_with_generate=False,
        logging_steps=100,
        fp16=True,
        dataloader_num_workers=2,
        dataloader_pin_memory=True,
        report_to="none",
        seed=SEED,
        gradient_checkpointing=True,
        optim="paged_adamw_8bit",      # 8-bit optimizer saves VRAM
    )

    eff_batch_p1 = P1_BATCH * P1_GRAD_ACCUM
    print(f"Phase 1 effective batch size: {eff_batch_p1}")

    trainer_p1 = Seq2SeqTrainer(
        model=model,
        args=args_p1,
        train_dataset=ds_p1_train,
        eval_dataset=ds_p1_val,
        data_collator=collator,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    )

    # --- Resume from checkpoint if available (survives Colab disconnects) ---
    p1_ckpt_dir = Path(p1_output_dir)
    p1_checkpoints = sorted(p1_ckpt_dir.glob("checkpoint-*")) if p1_ckpt_dir.exists() else []
    p1_resume = str(p1_checkpoints[-1]) if p1_checkpoints else None
    if p1_resume:
        print(f"Resuming Phase 1 from {p1_resume}")
    else:
        print("Phase 1: Starting from scratch...")

    trainer_p1.train(resume_from_checkpoint=p1_resume)
    model.save_pretrained(str(OUTPUT_DIR / "ckpt_phase1" / "best"))
    tokenizer.save_pretrained(str(OUTPUT_DIR / "ckpt_phase1" / "best"))
    print("Phase 1 complete")
else:
    print("Kaggle inference mode — skipping Phase 1 training")

## Phase 2: Sentence-Level Translation Training
Train on actual Akkadian→English sentence pairs with data augmentation (syllable dropout, word shuffle). Uses the vocabulary-warmed model from Phase 1.

In [ ]:
# ============ Cell 7: Phase 2 — Sentence Translation Training ============
# (Training only — skipped on Kaggle inference)

if not KAGGLE:
    # Data augmentation
    def augment_syllable_dropout(text, prob=0.1):
        words = text.split()
        out = []
        for w in words:
            if w.startswith('['):
                out.append(w)
                continue
            syls = w.split('-')
            if len(syls) > 1 and random.random() < prob:
                syls[random.randint(0, len(syls) - 1)] = 'x'
            out.append('-'.join(syls))
        return ' '.join(out)

    # Build Phase 2 data: SENT (3x upsample) + DOC (2x) + WORD (10% refresh)
    p2_sources, p2_targets = [], []

    for src, tgt, task in all_pairs:
        tagged_src = f"[{task}] {src}"
        if task == 'SENT':
            for _ in range(3):
                p2_sources.append(tagged_src)
                p2_targets.append(tgt)
                if random.random() < 0.5:
                    p2_sources.append(f"[SENT] {augment_syllable_dropout(src)}")
                    p2_targets.append(tgt)
        elif task == 'DOC':
            for _ in range(2):
                p2_sources.append(tagged_src)
                p2_targets.append(tgt)
        elif task == 'MINE':
            p2_sources.append(tagged_src)
            p2_targets.append(tgt)
        elif task == 'WORD' and random.random() < 0.1:
            p2_sources.append(tagged_src)
            p2_targets.append(tgt)

    # Shuffle
    combined = list(zip(p2_sources, p2_targets))
    random.shuffle(combined)
    p2_sources, p2_targets = zip(*combined) if combined else ([], [])
    p2_sources, p2_targets = list(p2_sources), list(p2_targets)

    # Deduplicate
    seen = set()
    dedup_src, dedup_tgt = [], []
    for s, t in zip(p2_sources, p2_targets):
        if s not in seen:
            seen.add(s)
            dedup_src.append(s)
            dedup_tgt.append(t)

    # Split
    p2_tr_s, p2_v_s, p2_tr_t, p2_v_t = train_test_split(
        dedup_src, dedup_tgt, test_size=0.05, random_state=SEED
    )
    print(f"Phase 2 training: {len(p2_tr_s)} train, {len(p2_v_s)} val")

    # Tokenize
    MAX_SRC_P2, MAX_TGT_P2 = 256, 256

    def tokenize_p2(examples):
        inputs = tokenizer(examples['source'], max_length=MAX_SRC_P2, padding='max_length', truncation=True)
        with tokenizer.as_target_tokenizer():
            labels = tokenizer(examples['target'], max_length=MAX_TGT_P2, padding='max_length', truncation=True)
        inputs['labels'] = labels['input_ids']
        return inputs

    ds_p2_train = Dataset.from_dict({'source': p2_tr_s, 'target': p2_tr_t})
    ds_p2_val = Dataset.from_dict({'source': p2_v_s, 'target': p2_v_t})
    ds_p2_train = ds_p2_train.map(tokenize_p2, batched=True, remove_columns=['source', 'target'])
    ds_p2_val = ds_p2_val.map(tokenize_p2, batched=True, remove_columns=['source', 'target'])

    # QLoRA on L4: 256-token sequences need smaller batch
    P2_BATCH = 2
    P2_EVAL_BATCH = 4
    P2_GRAD_ACCUM = 16   # effective batch = 2 × 16 = 32

    p2_output_dir = str(OUTPUT_DIR / "ckpt_phase2")

    args_p2 = Seq2SeqTrainingArguments(
        output_dir=p2_output_dir,
        num_train_epochs=15,
        per_device_train_batch_size=P2_BATCH,
        per_device_eval_batch_size=P2_EVAL_BATCH,
        gradient_accumulation_steps=P2_GRAD_ACCUM,
        learning_rate=5e-5,
        weight_decay=0.01,
        warmup_ratio=0.1,
        lr_scheduler_type="cosine_with_restarts",
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        predict_with_generate=False,
        logging_steps=50,
        fp16=True,
        dataloader_num_workers=2,
        dataloader_pin_memory=True,
        report_to="none",
        seed=SEED,
        label_smoothing_factor=0.1,
        gradient_checkpointing=True,
        optim="paged_adamw_8bit",
    )

    eff_batch_p2 = P2_BATCH * P2_GRAD_ACCUM
    print(f"Phase 2 effective batch size: {eff_batch_p2}")

    trainer_p2 = Seq2SeqTrainer(
        model=model,
        args=args_p2,
        train_dataset=ds_p2_train,
        eval_dataset=ds_p2_val,
        data_collator=collator,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
    )

    # --- Resume from checkpoint if available ---
    p2_ckpt_dir = Path(p2_output_dir)
    p2_checkpoints = sorted(p2_ckpt_dir.glob("checkpoint-*")) if p2_ckpt_dir.exists() else []
    p2_resume = str(p2_checkpoints[-1]) if p2_checkpoints else None
    if p2_resume:
        print(f"Resuming Phase 2 from {p2_resume}")
    else:
        print("Phase 2: Starting from scratch...")

    trainer_p2.train(resume_from_checkpoint=p2_resume)
    model.save_pretrained(str(OUTPUT_DIR / "ckpt_phase2" / "best"))
    tokenizer.save_pretrained(str(OUTPUT_DIR / "ckpt_phase2" / "best"))
    print("Phase 2 complete")
else:
    print("Kaggle inference mode — skipping Phase 2 training")

## Phase 3: Multi-Task Fine-Tuning with BLEU/chrF++ Optimization
Final phase with generation-based evaluation tracking the competition metric: `√(BLEU × chrF++)`

In [ ]:
# ============ Cell 8: Phase 3 — Fine-Tuning with BLEU Eval ============
# (Training only — skipped on Kaggle inference)

if not KAGGLE:
    import sacrebleu
    from math import sqrt

    def compute_metrics(eval_preds):
        """Compute geometric mean of BLEU and chrF++ (competition metric)."""
        preds, labels = eval_preds
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        decoded_preds = [p.strip() for p in tokenizer.batch_decode(preds, skip_special_tokens=True)]
        decoded_labels = [l.strip() for l in tokenizer.batch_decode(labels, skip_special_tokens=True)]

        bleu = sacrebleu.corpus_bleu(decoded_preds, [decoded_labels])
        chrf = sacrebleu.corpus_chrf(decoded_preds, [decoded_labels], word_order=2)
        geo = sqrt(max(bleu.score, 0.01) * max(chrf.score, 0.01))
        return {'bleu': bleu.score, 'chrf': chrf.score, 'geo_mean': geo}

    # Phase 3 data: all non-WORD pairs + 5% WORD refresh
    p3_sources, p3_targets = [], []
    for src, tgt, task in all_pairs:
        if task in ('SENT', 'DOC', 'MINE'):
            p3_sources.append(f"[{task}] {src}")
            p3_targets.append(tgt)
        elif task == 'WORD' and random.random() < 0.05:
            p3_sources.append(f"[WORD] {src}")
            p3_targets.append(tgt)

    combined3 = list(zip(p3_sources, p3_targets))
    random.shuffle(combined3)
    p3_sources, p3_targets = zip(*combined3) if combined3 else ([], [])
    p3_sources, p3_targets = list(p3_sources), list(p3_targets)

    p3_tr_s, p3_v_s, p3_tr_t, p3_v_t = train_test_split(
        p3_sources, p3_targets, test_size=0.05, random_state=SEED
    )

    ds_p3_train = Dataset.from_dict({'source': p3_tr_s, 'target': p3_tr_t})
    ds_p3_val = Dataset.from_dict({'source': p3_v_s, 'target': p3_v_t})
    ds_p3_train = ds_p3_train.map(tokenize_p2, batched=True, remove_columns=['source', 'target'])
    ds_p3_val = ds_p3_val.map(tokenize_p2, batched=True, remove_columns=['source', 'target'])

    # Phase 3: generation enabled → even more VRAM → smallest batch
    P3_BATCH = 1
    P3_EVAL_BATCH = 2
    P3_GRAD_ACCUM = 16   # effective batch = 1 × 16 = 16

    p3_output_dir = str(OUTPUT_DIR / "ckpt_phase3")

    args_p3 = Seq2SeqTrainingArguments(
        output_dir=p3_output_dir,
        num_train_epochs=10,
        per_device_train_batch_size=P3_BATCH,
        per_device_eval_batch_size=P3_EVAL_BATCH,
        gradient_accumulation_steps=P3_GRAD_ACCUM,
        learning_rate=1e-5,
        weight_decay=0.01,
        warmup_ratio=0.05,
        lr_scheduler_type="cosine",
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model="geo_mean",
        greater_is_better=True,
        predict_with_generate=True,
        generation_max_length=256,
        generation_num_beams=4,
        logging_steps=50,
        fp16=True,
        dataloader_num_workers=2,
        dataloader_pin_memory=True,
        report_to="none",
        seed=SEED,
        label_smoothing_factor=0.1,
        gradient_checkpointing=True,
        optim="paged_adamw_8bit",
    )

    eff_batch_p3 = P3_BATCH * P3_GRAD_ACCUM
    print(f"Phase 3 effective batch size: {eff_batch_p3}")

    trainer_p3 = Seq2SeqTrainer(
        model=model,
        args=args_p3,
        train_dataset=ds_p3_train,
        eval_dataset=ds_p3_val,
        data_collator=collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=4)],
    )

    # --- Resume from checkpoint if available ---
    p3_ckpt_dir = Path(p3_output_dir)
    p3_checkpoints = sorted(p3_ckpt_dir.glob("checkpoint-*")) if p3_ckpt_dir.exists() else []
    p3_resume = str(p3_checkpoints[-1]) if p3_checkpoints else None
    if p3_resume:
        print(f"Resuming Phase 3 from {p3_resume}")
    else:
        print("Phase 3: Starting from scratch...")

    trainer_p3.train(resume_from_checkpoint=p3_resume)

    # Save LoRA checkpoint (Phase 3 best)
    model.save_pretrained(str(OUTPUT_DIR / "ckpt_phase3" / "best"))
    tokenizer.save_pretrained(str(OUTPUT_DIR / "ckpt_phase3" / "best"))
    print("Phase 3 complete")
else:
    print("Kaggle inference mode — skipping Phase 3 training")

## Training Curves

In [ ]:
# ============ Cell 9: Training Curves ============
# (Only meaningful after training on Colab)

if not KAGGLE:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Phase 1
    p1_train = [l for l in trainer_p1.state.log_history if 'loss' in l]
    p1_eval = [l for l in trainer_p1.state.log_history if 'eval_loss' in l]
    if p1_train:
        axes[0].plot([l.get('step', i) for i, l in enumerate(p1_train)], [l['loss'] for l in p1_train], 'b-', alpha=0.7, label='Train')
    if p1_eval:
        axes[0].plot([l.get('step', i) for i, l in enumerate(p1_eval)], [l['eval_loss'] for l in p1_eval], 'r-o', label='Val')
    axes[0].set_title('Phase 1: Syllable→Meaning'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

    # Phase 2
    p2_train = [l for l in trainer_p2.state.log_history if 'loss' in l]
    p2_eval = [l for l in trainer_p2.state.log_history if 'eval_loss' in l]
    if p2_train:
        axes[1].plot([l.get('step', i) for i, l in enumerate(p2_train)], [l['loss'] for l in p2_train], 'b-', alpha=0.7, label='Train')
    if p2_eval:
        axes[1].plot([l.get('step', i) for i, l in enumerate(p2_eval)], [l['eval_loss'] for l in p2_eval], 'r-o', label='Val')
    axes[1].set_title('Phase 2: Sentence Translation'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

    # Phase 3
    p3_eval = [l for l in trainer_p3.state.log_history if 'eval_geo_mean' in l]
    if p3_eval:
        axes[2].plot(range(1, len(p3_eval)+1), [l['eval_bleu'] for l in p3_eval], 'g-o', label='BLEU')
        axes[2].plot(range(1, len(p3_eval)+1), [l['eval_chrf'] for l in p3_eval], 'b-s', label='chrF++')
        axes[2].plot(range(1, len(p3_eval)+1), [l['eval_geo_mean'] for l in p3_eval], 'r-^', lw=2, label='GeoMean')
    axes[2].set_title('Phase 3: BLEU/chrF++'); axes[2].legend(); axes[2].grid(True, alpha=0.3)

    plt.tight_layout(); plt.show()

    if p3_eval:
        best = max(p3_eval, key=lambda x: x['eval_geo_mean'])
        print(f"Best: BLEU={best['eval_bleu']:.2f}, chrF++={best['eval_chrf']:.2f}, GeoMean={best['eval_geo_mean']:.2f}")
else:
    print("Kaggle inference mode — no training curves to show")

## Merge LoRA & Save / Inference & Submission

**After training (Colab)**: Merge LoRA adapters into base model → save full merged model → download or push to HuggingFace Hub → upload to Kaggle as dataset.  
**On Kaggle**: Load the pre-merged model in 8-bit, generate translations, create `submission.csv`.

In [ ]:
# ============ Cell 10: Merge LoRA (Colab) / Inference & Submission ============

from tqdm.auto import tqdm

# ============================================================
# COLAB: Merge LoRA adapters → save full model for Kaggle
# ============================================================
if not KAGGLE:
    from peft import PeftModel

    print("Merging LoRA adapters into base model...")
    # merge_and_unload() folds LoRA weights into the base model
    merged_model = model.merge_and_unload()

    FINAL_DIR = str(OUTPUT_DIR / "final_merged")
    merged_model.save_pretrained(FINAL_DIR)
    tokenizer.save_pretrained(FINAL_DIR)

    # Calculate size
    import glob
    total_bytes = sum(os.path.getsize(f) for f in glob.glob(f"{FINAL_DIR}/**", recursive=True) if os.path.isfile(f))
    print(f"Merged model saved to: {FINAL_DIR}")
    print(f"Total size: {total_bytes / 1e9:.1f} GB")
    print()
    print("Next steps:")
    print("  1. Download the final_merged/ folder")
    print("  2. Create a Kaggle dataset named 'flan-t5-xxl-akkadian'")
    print("  3. Upload the folder contents to the dataset")
    print("  4. In your Kaggle submission notebook, add this dataset as input")
    print("  5. The inference cells below will automatically load from /kaggle/input/flan-t5-xxl-akkadian/final_merged")

# ============================================================
# INFERENCE (works on both Colab for validation and Kaggle for submission)
# ============================================================
# On Kaggle, model was already loaded in 8-bit in Cell 5
# On Colab, use the merged_model we just created
infer_model = model if KAGGLE else merged_model

def generate_batch(texts, max_length=256, num_beams=5, batch_size=4):
    """Batch generation with automatic device placement."""
    infer_model.eval()
    all_preds = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Translating"):
        batch = texts[i:i + batch_size]
        inputs = tokenizer(batch, return_tensors="pt", max_length=256, padding=True, truncation=True)
        # With device_map="auto", inputs go to the model's first device
        first_device = next(infer_model.parameters()).device
        inputs = {k: v.to(first_device) for k, v in inputs.items()}
        with torch.no_grad():
            out = infer_model.generate(
                **inputs,
                max_length=max_length,
                num_beams=num_beams,
                early_stopping=True,
                no_repeat_ngram_size=3,
                length_penalty=1.0,
            )
        all_preds.extend(tokenizer.batch_decode(out, skip_special_tokens=True))
    return all_preds

def post_process(pred):
    pred = re.sub(r'\s+', ' ', pred).strip()
    if not pred or len(pred) < 3:
        pred = "the tablet is broken"
    sentences = pred.split('. ')
    sentences = [s[0].upper() + s[1:] if len(s) > 1 else s for s in sentences]
    pred = '. '.join(sentences)
    if pred and pred[-1] not in '.!?':
        pred += '.'
    return pred

# ---- Generate translations ----
test_sources = [f"[SENT] {clean_transliteration(str(r['source']))}" for _, r in test_df.iterrows()]

# Batch size: 4 for 8-bit xxl (conservative for beam search VRAM)
INFER_BATCH = 4 if KAGGLE else 2
raw_preds = generate_batch(test_sources, max_length=256, num_beams=5, batch_size=INFER_BATCH)
final_preds = [post_process(p) for p in raw_preds]

# ---- Create submission ----
submission = pd.DataFrame({'id': test_df['id'], 'target': final_preds})
sub_path = str(OUTPUT_DIR / 'submission.csv')
submission.to_csv(sub_path, index=False)

print(f"\nSubmission saved: {sub_path}")
print(f"   Predictions: {len(submission)}")

# Show samples
for i in range(min(5, len(submission))):
    print(f"\n  SRC: {test_sources[i][7:][:80]}...")
    print(f"  TGT: {final_preds[i][:120]}")

# Sanity checks
assert len(submission) == len(test_df), "Row count mismatch!"
assert submission['target'].isna().sum() == 0, "NaN predictions!"
print(f"\nAll {len(submission)} predictions ready. Submit to Kaggle!")